In [1]:
from unsloth import FastModel
import torch
import numpy as np
import random

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
INFO 05-23 13:27:35 [__init__.py:235] Automatically detected platform cuda.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [2]:
# Modelo
SUFFIX = "ckpt-200"
# CHECKPOINT_DIR = f"/exp_local/kenzosaki/models/ai_events_gemma_3_4b_clm_r128_lr1e-4_no_json/checkpoint-{SUFFIX.split('-')[1]}"
CHECKPOINT_DIR = f"/exp_local/kenzosaki/models/ai_events_gemma_3_4b_dpo/checkpoint-{SUFFIX.split('-')[1]}"
DEVICE = "cuda"
MAX_SEQ_LENGTH = 512 # nao temm pq ser muito grande
EVAL_BS = 4

In [3]:
# Dados
TEST_EDGES = "../data/labels/small_directed_mad3.5/link_prediction_edges.json"
GRAPH_PATH = "../data/processed/directed_mad3.5_ai_news_event_graph.pkl"
SAMPLE_SIZE = 1000
USE_JSON_STRS = False

In [4]:
SEED = 2026
np.random.seed(SEED)
random.seed(SEED)

# Carregando dados

In [5]:
import json
import pickle
import networkx as nx
from glm_based_event_analysis.utils.graph import get_node_metadata, remove_edges_from_graph

In [6]:
with open(TEST_EDGES, "r") as f:
    test_edges = json.load(f)

with open(GRAPH_PATH, "rb") as f:
    G = pickle.load(f)

In [7]:
G_train = remove_edges_from_graph(G, test_edges)

# Preparando exemplos do teste para avaliação

In [ ]:
import json
from glm_based_event_analysis.link_prediction.datasets import EVENT_STR_TEMPLATE

In [9]:
def prepare_edge_json_strs(edges: list[tuple[str, str]], G: nx.Graph) -> list[str]:
    """
    Prepares a list of JSON strings for a given list of edges and a graph.
    Each JSON string contains the metadata of the two nodes connected by the edge.
    
    Args:
        edges (list[tuple[str, str]]): A list of edges, where each edge is represented as a tuple (u, v).
        G (nx.Graph): The graph containing the nodes and their metadata.
    
    Returns:
        list[str]: A list of JSON strings, where each string contains the metadata of the two nodes connected by an edge.
    """
    edge_json_strs = []
    
    for edge in edges:
        u = edge[0]
        v = edge[1]
        u_data = get_node_metadata(G, u)
        v_data = get_node_metadata(G, v)
        
        edge_json_str = json.dumps([u_data, v_data], indent=None)
        edge_json_strs.append(edge_json_str[:-2]) # desconsiderando ultimos caracteres '\n}' do json pois nao influenciam na predição.
    
    return edge_json_strs

def prepare_simple_edge_strs(edges: list[tuple[str, str]], G: nx.Graph) -> list[str]:
    """
    Prepares a list of simple edge strings for a given list of edges and a graph.
    Each string contains the IDs of the two nodes connected by the edge.
    
    Args:
        edges (list[tuple[str, str]]): A list of edges, where each edge is represented as a tuple (u, v).
        G (nx.Graph): The graph containing the nodes and their metadata.
    
    Returns:
        list[str]: A list of simple edge strings, where each string contains the IDs of the two nodes connected by an edge.
    """
    edge_strs = []
    
    for edge in edges:
        u = edge[0]
        v = edge[1]

        u_metadata = get_node_metadata(G, u)
        v_metadata = get_node_metadata(G, v)
        
        u_str = EVENT_STR_TEMPLATE.format(
            id=u,
            who=u_metadata.get("who", ""),
            what=u_metadata.get("what", ""),
            when=u_metadata.get("when", ""),
            where=u_metadata.get("where", ""),
            why=u_metadata.get("why", ""),
            how=u_metadata.get("how", "")
        )

        v_str = EVENT_STR_TEMPLATE.format(
            id=v,
            who=v_metadata.get("who", ""),
            what=v_metadata.get("what", ""),
            when=v_metadata.get("when", ""),
            where=v_metadata.get("where", ""),
            why=v_metadata.get("why", ""),
            how=v_metadata.get("how", "")
        )

        edge_str = f"{u_str}\n{v_str}"

        edge_strs.append(edge_str)
    
    return edge_strs

In [10]:
test_edges_ids = [(edge_info['u'], edge_info['v']) for edge_info in test_edges]
test_labels = [edge_info['label'] for edge_info in test_edges]

In [11]:
test_edges_json_strs = prepare_edge_json_strs(test_edges_ids, G)

In [12]:
print(test_edges_json_strs[0])

[{"what": "Nebius secures a $4.3 billion debt raise to fund its AI initiatives.", "where": "United States", "when": "2026-03-23-07:00:00", "who": "Nebius", "how": "Through a $4.3 billion debt raise.", "why": "To gain a competitive advantage in the AI race.", "event_id": "9107_ai_nebius_debt_2026-03-23"}, {"what": "Mistral secures $830 million in debt financing for an AI data center in Paris utilizing Nvidia chips.", "where": "France", "when": "2026-03-30-07:00:00", "who": "Mistral, Nvidia", "how": "Through the acquisition of $830 million in debt financing.", "why": "To expand AI infrastructure and processing capabilities.", "event_id": "10850_mistral_debt_paris_2026-03-30"


# Amostrando arestas positivas

In [13]:
import numpy as np

In [14]:
selected_edge_ids = np.random.choice(G_train.number_of_edges(), size=SAMPLE_SIZE, replace=False)

In [15]:
train_edges = list(G_train.edges())
sampled_edges = [train_edges[i] for i in selected_edge_ids]

In [16]:
sampled_edges[:5]

[('3023_windows_hudson_valley_2026-03-06',
  '8168_microsoft_windows_ai_2026-03-07'),
 ('1560_ai_moratorium_data_2026-03-26', '12935_ai_congress_trump_2026-03-28'),
 ('3719_education_digital_learning_2026-03-02',
  '4321_python_decorators_llm_2026-03-06'),
 ('5968_openai_chatgpt_ads_2026-03-12',
  '6145_openai_venture_equity_2026-03-16'),
 ('9108_xbox_partners_preview_2026-03-23',
  '11198_chatbot_games_trend_2026-03-27')]

In [17]:
if USE_JSON_STRS:
    sampled_edges_json_strs = prepare_edge_json_strs(sampled_edges, G_train)
else:
    sampled_edges_json_strs = prepare_simple_edge_strs(sampled_edges, G_train)

In [18]:
print(sampled_edges_json_strs[0])

<event>
When: 2026-03-06-08:00:00
What: Microsoft is developing Windows 12, codenamed 'Hudson Valley', with anticipated improvements and new features.
Who: Microsoft
Why: To enhance the user experience and maintain market competitiveness in the operating system landscape.
Where: United States
How: Through internal development processes, software engineering, and testing phases, culminating in a public release.
ID: 3023_windows_hudson_valley_2026-03-06
</event>

<event>
When: 2026-03-07-08:00:00
What: Microsoft is reportedly planning a significant integration of artificial intelligence into Windows 12.
Who: Microsoft, Windows users
Why: To enhance user experience and remain competitive in the operating system market.
Where: United States
How: Through a drastic integration of AI features within the Windows 12 operating system.
ID: 8168_microsoft_windows_ai_2026-03-07
</event>



# Carregando modelo treinado

In [19]:
model, tokenizer = FastModel.from_pretrained(
    model_name = CHECKPOINT_DIR, # Can be local or HF repo
    max_seq_length = 512,
    load_in_4bit = True,
    device_map = DEVICE,
    fast_inference=False
)


==((====))==  Unsloth 2026.5.2: Fast Gemma3 patching. Transformers: 4.57.2. vLLM: 0.10.0+cu126.
   \\   /|    NVIDIA RTX A5000. Num GPUs = 1. Max memory: 23.679 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.1+cu126. CUDA: 8.6. CUDA Toolkit: 12.6. Triton: 3.3.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.31. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [20]:
model.device

device(type='cuda', index=0)

In [21]:
#model = model.to(torch.float)
model = FastModel.for_inference(model)

In [22]:
if hasattr(tokenizer, "tokenizer"):
    print("Extracting the tokenizar from Processor.")
    tokenizer = tokenizer.tokenizer

Extracting the tokenizar from Processor.


# Calculo de perplexidades

In [23]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from tqdm import tqdm

def compute_perplexity(
    texts: list[str],
    max_seq_length: int,
    tokenizer: AutoTokenizer,
    model: AutoModelForCausalLM,
    device: torch.device,
    bs: int = 256) -> tuple[list[str], float]:

    perplexities = []
    loss = torch.nn.CrossEntropyLoss(reduction='none')

    with torch.no_grad():

      for i in tqdm(range(0, len(texts), bs), desc="Remaining batches"):

        batch_texts = texts[i:i+bs]

        batch_model_inp = tokenizer.batch_encode_plus(
            batch_texts,
            add_special_tokens=False,
            return_tensors='pt',
            padding='max_length',
            max_length=max_seq_length,
            truncation=True,
            return_attention_mask=True,
        )

        batch_model_inp = batch_model_inp.to(device)
        # geração das caminhadas como ids de embedding
        output = model(**batch_model_inp).logits

        # adaptado de: https://huggingface.co/spaces/evaluate-metric/perplexity/blob/main/perplexity.py
        shift_logits = output[..., :-1, :].contiguous()
        shift_labels = batch_model_inp['input_ids'][..., 1:].contiguous()
        shift_attention_mask_batch = batch_model_inp['attention_mask'][..., 1:].contiguous()

        perplexity_batch = torch.exp(
            (loss(shift_logits.transpose(1, 2), shift_labels) * shift_attention_mask_batch).sum(1)
            / shift_attention_mask_batch.sum(1)
        )

        perplexities.extend(perplexity_batch.tolist())

    # retorna sequências de identificador
    return perplexities

In [24]:
# arestas de teste
test_edge_perplexities = compute_perplexity(
    texts = test_edges_json_strs,
    max_seq_length = MAX_SEQ_LENGTH,
    tokenizer = tokenizer,
    model = model,
    device = model.device,
    bs = EVAL_BS
)

Remaining batches: 100%|██████████| 178/178 [02:01<00:00,  1.47it/s]


In [25]:
test_edge_perplexities[:5]

[123.0, 127.0, 148.0, 79.5, 60.0]

In [26]:
# sampled true edges
sampled_edges_perplexities = compute_perplexity(
    texts = sampled_edges_json_strs,
    max_seq_length = MAX_SEQ_LENGTH,
    tokenizer = tokenizer,
    model = model,
    device = model.device,
    bs = EVAL_BS
)

Remaining batches: 100%|██████████| 250/250 [02:45<00:00,  1.51it/s]


In [27]:
sampled_edges_perplexities[:5]

[11.8125, 42.5, 51.25, 11.8125, 40.0]

In [28]:
# salvando os resultados para analise
results_dict = {
    "test_perplexities": test_edge_perplexities,
    "test_labels": test_labels,
    "sampled_edges_perplexities": sampled_edges_perplexities,
}

with open(f"{CHECKPOINT_DIR}/clm_experiment.pkl", "wb") as f:
    pickle.dump(results_dict, f)

# Repetindo o experimento adicionando contexto adicional

In [31]:
from glm_based_event_analysis.random_walks.sampler import RandomWalkSampler
import random

In [29]:
# obtendo um grafo direcionado reverso para amostrar eventos anteriores a aresta de origem
reverse_G_train = G_train.reverse()

In [32]:
walk_size = 3 # 4 contando o nó de origem
num_walks_per_node = 3
sampler = RandomWalkSampler("none", walk_size, num_walks_per_node)

In [ ]:
# obtendo o contexto adicional (eventos anteriores a u)
eval_info = []

for edge_info in test_edges:
    u = edge_info["u"]
    v = edge_info["v"]
    label = edge_info["label"]

    # o código pega apenas *uma* caminhada. porém, poderiam ser várias.
    prior_events = sampler.sample_random_walk(u, reverse_G_train)

    eval_info.append({
        "u": u,
        "v": v,
        "label": label,
        "prior_events": prior_events
    })

In [ ]:
# invertendo o sentido das caminhadas e preparando a avaliação
walks = []
labels = []

for eval_rw in eval_info:
    event_history = eval_rw["prior_events"]
    event_history.reverse() 
    event_history.append(eval_rw["v"]) # a ultima aresta *não* é vista no treino, e pode ser positiva ou negativa

    walks.append(event_history)
    labels.append(eval_rw["label"])

In [35]:
random.choice(walks)

[np.str_('8191_xbox_microsoft_reveal_2026-03-07'),
 np.str_('8660_xbox_ai_companion_2026-03-14'),
 np.str_('8875_xbox_controller_sale_2026-03-19'),
 np.str_('9108_xbox_partners_preview_2026-03-23'),
 '7478_google_tv_gemini_2026-03-24',
 '7800_google_pixel_habits_2026-03-30']

In [36]:
walks_with_metadata = []
for walk in walks:
    walk_metadata = []
    for node in walk:
        node_metadata = get_node_metadata(G, node)
        walk_metadata.append(node_metadata)
    walks_with_metadata.append(walk_metadata)

In [37]:
random.choice(walks_with_metadata)

[{'what': "UMG, Concord, and ABKCO filed a motion requesting a judge to rule that Anthropic infringed their copyrights and reject the AI 'fair use' argument.",
  'where': 'United States',
  'when': '2026-03-24-07:00:00',
  'who': 'UMG, Concord, ABKCO, Anthropic',
  'how': "By filing a motion with a judge to rule on copyright infringement and the validity of the 'fair use' argument.",
  'why': "To prevent Anthropic from using copyrighted material without permission and to challenge the 'fair use' defense.",
  'event_id': np.str_('10587_copyright_anthropic_infringement_2026-03-24')},
 {'what': 'A US judge is questioning a ban affecting Anthropic, a US-based AI company.',
  'where': 'United States',
  'when': '2026-03-25-07:00:00',
  'who': 'US judge, Anthropic',
  'how': "Through a legal review and questioning of the ban's implementation.",
  'why': "To assess the legality and potential impact of the ban on Anthropic's operations.",
  'event_id': np.str_('10624_anthropic_ban_judge_2026-0

In [38]:
if USE_JSON_STRS:
    final_eval_rws = [json.dumps(walk, indent=None) for walk in walks_with_metadata]
else:
    final_eval_rws = []
    for walk in walks_with_metadata:
        walk_str = ""
        for node_metadata in walk:
            node_str = EVENT_STR_TEMPLATE.format(
                id=node_metadata.get("event_id", ""),
                who=node_metadata.get("who", ""),
                what=node_metadata.get("what", ""),
                when=node_metadata.get("when", ""),
                where=node_metadata.get("where", ""),
                why=node_metadata.get("why", ""),
                how=node_metadata.get("how", "")
            )
            walk_str += node_str + "\n"
        final_eval_rws.append(walk_str)

In [39]:
print(final_eval_rws[0])

<event>
When: 2026-03-07-08:00:00
What: OpenAI’s head of robotics resigned due to a company agreement with the Pentagon.
Who: OpenAI’s head of robotics, OpenAI, Pentagon
Why: Disagreement over the ethical implications of the Pentagon deal.
Where: United States
How: The head of robotics submitted a resignation following internal discussions about the partnership.
ID: 5715_robotics_pentagon_deal_2026-03-07
</event>

<event>
When: 2026-03-08-08:00:00
What: OpenAI Robotics Chief Caitlin Kalinowski resigns over a rushed Pentagon defense deal.
Who: Caitlin Kalinowski, OpenAI, Pentagon
Why: Concerns over the rushed nature of a defense deal.
Where: United States
How: Kalinowski resigned from her position at OpenAI Robotics.
ID: 5762_openai_pentagon_deal_2026-03-08
</event>

<event>
When: 2026-03-12-07:00:00
What: Musk unveils a joint Tesla-xAI project named "Macrohard" to disrupt the software industry.
Who: Elon Musk, Tesla, xAI
Why: To disrupt the existing software industry landscape.
Where: 

In [40]:
test_perplexities_with_history = compute_perplexity(
    final_eval_rws,
    max_seq_length = MAX_SEQ_LENGTH*2,
    tokenizer = tokenizer,  
    model = model,
    device = DEVICE,
    bs = 3
)

Remaining batches: 100%|██████████| 238/238 [03:42<00:00,  1.07it/s]


In [41]:
logs = {
    "test_perplexities_with_history": test_perplexities_with_history,
    "labels": labels,
}

with open(f"{CHECKPOINT_DIR}/event_history.pkl", "wb") as f:
    pickle.dump(logs, f)